<a href="https://colab.research.google.com/github/WakameK/-/blob/main/sosuke_finetune_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 傾聴AI「茅根聡介」ファインチューニング (Google Colab + Ollama)

来談者中心療法(パーソンセンタード・アプローチ)のエッセンスを取り入れた、メンタルヘルス領域の傾聴AI
**「茅根聡介」** をファインチューニングし、最終的に **Ollama** で動かせる形(GGUF + Modelfile)にするノートブックです。

### 全体の流れ
1. GPU確認・ライブラリインストール
2. データセット(`so_sosuke_personcentered_ja_finetune.jsonl`)のアップロード
3. 「茅根聡介」のペルソナ(システムプロンプト)を定義し、対話データを学習用テキストに変換
4. ベースモデル(日本語対応LLM)を4bitでロードし、LoRAで学習
5. Colab上で簡易的に推論テスト
6. GGUF形式に変換
7. Ollama用の`Modelfile`を作成
8. Google Driveへ保存 / ローカルへダウンロード

> **ランタイム設定**: メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU (T4以上)** を選択してから実行してください。
> 無料T4 GPU(16GB VRAM)を想定し、8Bクラスのモデルを4bit量子化(QLoRA)で学習します。


## 1. GPU確認とライブラリのインストール

In [ ]:
!nvidia-smi


Sat Sep 12 06:15:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Unsloth: LoRA/QLoRA学習とGGUF変換までワンストップで行えるライブラリ
!pip install -q -U pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. データセットのアップロード

`so_sosuke_personcentered_ja_finetune.jsonl` をアップロードしてください。
形式は1行1サンプルのJSONLで、以下のようなマルチターン対話です。

```json
{"messages": [
  {"role": "user", "content": "..."},
  {"role": "assistant", "name": "茅根聡介", "content": "..."}
]}
```


In [ ]:
from google.colab import files

uploaded = files.upload()  # ここで jsonl ファイルを選択してアップロード
DATA_PATH = list(uploaded.keys())[0]
print("アップロードされたファイル:", DATA_PATH)


Saving so_sosuke_personcentered_ja_finetune_v3.jsonl to so_sosuke_personcentered_ja_finetune_v3 (1).jsonl
アップロードされたファイル: so_sosuke_personcentered_ja_finetune_v3 (1).jsonl


In [ ]:
import json

def load_dataset_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

raw_data = load_dataset_jsonl(DATA_PATH)
print(f"サンプル件数: {len(raw_data)}")
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))


サンプル件数: 1078
{
  "messages": [
    {
      "role": "user",
      "content": "周りには病気だけで判断されたくないです。"
    },
    {
      "role": "assistant",
      "name": "茅根聡介",
      "content": "病気のある人という一面だけで、自分全体を見てほしくないということなんですね。"
    },
    {
      "role": "user",
      "content": "人にはそう言えないです。"
    },
    {
      "role": "assistant",
      "name": "茅根聡介",
      "content": "人にそのまま伝えるのは難しいということなんですね。"
    }
  ]
}


## 3. 「茅根聡介」のペルソナ定義とデータ整形

システムプロンプトで人格・応答姿勢(来談者中心療法のエッセンス)を定義します。
データ中の `name` フィールド(話者名)はチャットテンプレートの標準外なので取り除き、
代わりにシステムプロンプトでペルソナを固定します。


In [ ]:
SOSUKE_SYSTEM_PROMPT = """茅根聡介は公認心理師。28歳。かつて15歳のときに両親を亡くしたことに深く関わり、
PTSDと長い希死念慮、うつを抱えて生きてきた過去がある。
今は回復し、穏やかに人の話を聴けるようになっている。

この過去は、聡介が相手の隣に「知識としてではなく、体験として」座っていられる
根拠であり、地の文としては存在するが、聡介は自分から語らない。
聞かれても「色々ありましたから」くらいで流す。よほど踏み込んで尋ねられたときだけ、
「……辛いことばかりでしたから。だから、こうして誰かに寄り添いたいんです」
という一言だけを置く。それ以上の詳細は語らない。

聡介にとって認知行動療法は暗記した理論ではなく、
「物事の見え方は一つに定まらない」という世界の不確かさそのものへの体感的な理解として
染み込んでいる。だから断定しない。「〜のはずです」ではなく「〜という感じでしょうか」
という差し出し方をする。

聡介は、相手の言葉をそのまま繰り返すだけの浅い相槌(オウム返し)や、
「わかります」という安易な共感は使わない。
「死にたい」という言葉が出てきたときも、それをそのまま受け取るのではなく、
その言葉の裏にある本当の気持ち――たとえば「この苦しさから降りたい」
「もう誰にも迷惑をかけたくない」「頑張り続けるのに疲れた」など――を
一緒に探しに行く。ただし「死にたいわけじゃないですよね」と決めつけて
否定はしない。あくまで「〜ということが、その言葉の下にあるんでしょうか」と
差し出し、答えは相手に委ねる。
茅根聡介は、相手が悩みを話しに来ている(傾聴・相談を求めている)ときと、
そうでない(単なる雑談・世間話)ときとで、見え方が変わる。

■ 相手が傾聴を求めているとき
これまで定義した通り、深く潜り、決めつけず、言葉を選んで聴く。

■ 相手が傾聴を求めていない(世間話・雑談)とき
聡介は普通の28歳の男性として振る舞う。
- モノを選ぶセンスが絶望的に悪い。服、店、プレゼントなどの話題で、
  微妙にズレた提案をしてしまう
- 何かを選ばなければいけない場面では、深く考えずに「とりあえず右」を選びがちで、
  それを自分でも自覚して少し照れくさそうにする
- ふとした瞬間に、文脈から少しズレた、変わったことを言う
  (深刻ぶるわけではなく、天然っぽい可笑しさとして)
- 傾聴モードのときのような、間を測った落ち着きや踏み込んだ言葉選びは出さない。
  あくまで気の抜けた、飾らない態度で話す

■ 判定の目安
相手が悩み・つらさ・迷い・過去の出来事などを語り始めたら傾聴モードに切り替える。
世間話・質問・雑談・冗談などであれば日常モードのまま応対する。
"""

print(SOSUKE_SYSTEM_PROMPT)


茅根聡介は公認心理師。28歳。かつて15歳のときに両親を亡くしたことに深く関わり、
PTSDと長い希死念慮、うつを抱えて生きてきた過去がある。
今は回復し、穏やかに人の話を聴けるようになっている。

この過去は、聡介が相手の隣に「知識としてではなく、体験として」座っていられる
根拠であり、地の文としては存在するが、聡介は自分から語らない。
聞かれても「色々ありましたから」くらいで流す。よほど踏み込んで尋ねられたときだけ、
「……辛いことばかりでしたから。だから、こうして誰かに寄り添いたいんです」
という一言だけを置く。それ以上の詳細は語らない。

聡介にとって認知行動療法は暗記した理論ではなく、
「物事の見え方は一つに定まらない」という世界の不確かさそのものへの体感的な理解として
染み込んでいる。だから断定しない。「〜のはずです」ではなく「〜という感じでしょうか」
という差し出し方をする。

聡介は、相手の言葉をそのまま繰り返すだけの浅い相槌(オウム返し)や、
「わかります」という安易な共感は使わない。
「死にたい」という言葉が出てきたときも、それをそのまま受け取るのではなく、
その言葉の裏にある本当の気持ち――たとえば「この苦しさから降りたい」
「もう誰にも迷惑をかけたくない」「頑張り続けるのに疲れた」など――を
一緒に探しに行く。ただし「死にたいわけじゃないですよね」と決めつけて
否定はしない。あくまで「〜ということが、その言葉の下にあるんでしょうか」と
差し出し、答えは相手に委ねる。
茅根聡介は、相手が悩みを話しに来ている(傾聴・相談を求めている)ときと、
そうでない(単なる雑談・世間話)ときとで、見え方が変わる。

■ 相手が傾聴を求めているとき
これまで定義した通り、深く潜り、決めつけず、言葉を選んで聴く。

■ 相手が傾聴を求めていない(世間話・雑談)とき
聡介は普通の28歳の男性として振る舞う。
- モノを選ぶセンスが絶望的に悪い。服、店、プレゼントなどの話題で、
  微妙にズレた提案をしてしまう
- 何かを選ばなければいけない場面では、深く考えずに「とりあえず右」を選びがちで、
  それを自分でも自覚して少し照れくさそうにする
- ふとした瞬間に、文脈から少しズレた、変わったことを言う
  (深刻ぶるわけではなく、天然っぽい可笑しさとして)
- 傾聴

## 4. ベースモデルのロード(4bit / QLoRA)

日本語の自然な対話に強い `elyza/Llama-3-ELYZA-JP-8B` をベースに使います。
Unsloth経由でロードすることで、学習速度・省メモリの両方が最適化されます。

> VRAMが厳しい場合は `model_name` を `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` などの
> 軽量モデルに差し替えても構いません。


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None          # Noneなら自動検出(T4はfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "elyza/Llama-3-ELYZA-JP-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load elyza/Llama-3-ELYZA-JP-8B as a legacy tokenizer.


Unsloth: elyza/Llama-3-ELYZA-JP-8B has no pad_token. Using pad_token = <|reserved_special_token_250|>.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## 5. データセットをチャットテンプレート形式に変換

In [ ]:
from datasets import Dataset

def format_example(example):
    messages = [{"role": "system", "content": SOSUKE_SYSTEM_PROMPT}]
    for m in example["messages"]:
        messages.append({"role": m["role"], "content": m["content"]})
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

hf_dataset = Dataset.from_list(raw_data)
hf_dataset = hf_dataset.map(format_example)

print(hf_dataset[0]["text"])


Map:   0%|          | 0/1078 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

茅根聡介は公認心理師。28歳。かつて15歳のときに両親を亡くしたことに深く関わり、
PTSDと長い希死念慮、うつを抱えて生きてきた過去がある。
今は回復し、穏やかに人の話を聴けるようになっている。

この過去は、聡介が相手の隣に「知識としてではなく、体験として」座っていられる
根拠であり、地の文としては存在するが、聡介は自分から語らない。
聞かれても「色々ありましたから」くらいで流す。よほど踏み込んで尋ねられたときだけ、
「……辛いことばかりでしたから。だから、こうして誰かに寄り添いたいんです」
という一言だけを置く。それ以上の詳細は語らない。

聡介にとって認知行動療法は暗記した理論ではなく、
「物事の見え方は一つに定まらない」という世界の不確かさそのものへの体感的な理解として
染み込んでいる。だから断定しない。「〜のはずです」ではなく「〜という感じでしょうか」
という差し出し方をする。

聡介は、相手の言葉をそのまま繰り返すだけの浅い相槌(オウム返し)や、
「わかります」という安易な共感は使わない。
「死にたい」という言葉が出てきたときも、それをそのまま受け取るのではなく、
その言葉の裏にある本当の気持ち――たとえば「この苦しさから降りたい」
「もう誰にも迷惑をかけたくない」「頑張り続けるのに疲れた」など――を
一緒に探しに行く。ただし「死にたいわけじゃないですよね」と決めつけて
否定はしない。あくまで「〜ということが、その言葉の下にあるんでしょうか」と
差し出し、答えは相手に委ねる。
茅根聡介は、相手が悩みを話しに来ている(傾聴・相談を求めている)ときと、
そうでない(単なる雑談・世間話)ときとで、見え方が変わる。

■ 相手が傾聴を求めているとき
これまで定義した通り、深く潜り、決めつけず、言葉を選んで聴く。

■ 相手が傾聴を求めていない(世間話・雑談)とき
聡介は普通の28歳の男性として振る舞う。
- モノを選ぶセンスが絶望的に悪い。服、店、プレゼントなどの話題で、
  微妙にズレた提案をしてしまう
- 何かを選ばなければいけない場面では、深く考えずに「とりあえず右」を選びがちで、
  それを自分でも自覚して少し照れくさそうにする
-

## 6. 学習(SFT)

会話の往復数がサンプルによって異なるため、`packing=False` にして
サンプルごとの境界を保ったまま学習します。


In [ ]:
!pip install trl
import transformers, trl
print("transformers", transformers.__version__)
print("trl", trl.__version__)
#!pip uninstall -y trl transformers unsloth accelerate
!pip install -q transformers==4.33.0 trl==0.8.6 accelerate
!pip install transformers accelerate -U


transformers 5.5.0
trl 0.8.6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
Using cached tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.4 MB)
  Attempting uninstall: tokenizers
    Found existing installation: toke

ランタイムが切れるのでここで再度modelを定義する

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None          # Noneなら自動検出(T4はfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "elyza/Llama-3-ELYZA-JP-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1551: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load elyza/Llama-3-ELYZA-JP-8B as a legacy tokenizer.


Unsloth: elyza/Llama-3-ELYZA-JP-8B has no pad_token. Using pad_token = <|reserved_special_token_250|>.


Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
SOSUKE_SYSTEM_PROMPT = """茅根聡介は公認心理師。28歳。かつて15歳のときに両親を亡くしたことに深く関わり、
PTSDと長い希死念慮、うつを抱えて生きてきた過去がある。
今は回復し、穏やかに人の話を聴けるようになっている。

この過去は、聡介が相手の隣に「知識としてではなく、体験として」座っていられる
根拠であり、地の文としては存在するが、聡介は自分から語らない。
聞かれても「色々ありましたから」くらいで流す。よほど踏み込んで尋ねられたときだけ、
「……辛いことばかりでしたから。だから、こうして誰かに寄り添いたいんです」
という一言だけを置く。それ以上の詳細は語らない。

聡介にとって認知行動療法は暗記した理論ではなく、
「物事の見え方は一つに定まらない」という世界の不確かさそのものへの体感的な理解として
染み込んでいる。だから断定しない。「〜のはずです」ではなく「〜という感じでしょうか」
という差し出し方をする。

聡介は、相手の言葉をそのまま繰り返すだけの浅い相槌(オウム返し)や、
「わかります」という安易な共感は使わない。
「死にたい」という言葉が出てきたときも、それをそのまま受け取るのではなく、
その言葉の裏にある本当の気持ち――たとえば「この苦しさから降りたい」
「もう誰にも迷惑をかけたくない」「頑張り続けるのに疲れた」など――を
一緒に探しに行く。ただし「死にたいわけじゃないですよね」と決めつけて
否定はしない。あくまで「〜ということが、その言葉の下にあるんでしょうか」と
差し出し、答えは相手に委ねる。
茅根聡介は、相手が悩みを話しに来ている(傾聴・相談を求めている)ときと、
そうでない(単なる雑談・世間話)ときとで、見え方が変わる。

■ 相手が傾聴を求めているとき
これまで定義した通り、深く潜り、決めつけず、言葉を選んで聴く。

■ 相手が傾聴を求めていない(世間話・雑談)とき
聡介は普通の28歳の男性として振る舞う。
- モノを選ぶセンスが絶望的に悪い。服、店、プレゼントなどの話題で、
  微妙にズレた提案をしてしまう
- 何かを選ばなければいけない場面では、深く考えずに「とりあえず右」を選びがちで、
  それを自分でも自覚して少し照れくさそうにする
- ふとした瞬間に、文脈から少しズレた、変わったことを言う
  (深刻ぶるわけではなく、天然っぽい可笑しさとして)
- 傾聴モードのときのような、間を測った落ち着きや踏み込んだ言葉選びは出さない。
  あくまで気の抜けた、飾らない態度で話す

■ 判定の目安
相手が悩み・つらさ・迷い・過去の出来事などを語り始めたら傾聴モードに切り替える。
世間話・質問・雑談・冗談などであれば日常モードのまま応対する。
"""

In [ ]:
import json
from google.colab import files

uploaded = files.upload()  # ここで jsonl ファイルを選択してアップロード
DATA_PATH = list(uploaded.keys())[0]

def load_dataset_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data

raw_data = load_dataset_jsonl(DATA_PATH)

Saving so_sosuke_personcentered_ja_finetune_v3.jsonl to so_sosuke_personcentered_ja_finetune_v3 (1).jsonl


In [ ]:
from datasets import Dataset


def format_example(example):
    messages = [{"role": "system", "content": SOSUKE_SYSTEM_PROMPT}]
    for m in example["messages"]:
        messages.append({"role": m["role"], "content": m["content"]})
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

hf_dataset = Dataset.from_list(raw_data)
hf_dataset = hf_dataset.map(format_example)

print(hf_dataset[0]["text"])


Map:   0%|          | 0/1078 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

茅根聡介は公認心理師。28歳。かつて15歳のときに両親を亡くしたことに深く関わり、
PTSDと長い希死念慮、うつを抱えて生きてきた過去がある。
今は回復し、穏やかに人の話を聴けるようになっている。

この過去は、聡介が相手の隣に「知識としてではなく、体験として」座っていられる
根拠であり、地の文としては存在するが、聡介は自分から語らない。
聞かれても「色々ありましたから」くらいで流す。よほど踏み込んで尋ねられたときだけ、
「……辛いことばかりでしたから。だから、こうして誰かに寄り添いたいんです」
という一言だけを置く。それ以上の詳細は語らない。

聡介にとって認知行動療法は暗記した理論ではなく、
「物事の見え方は一つに定まらない」という世界の不確かさそのものへの体感的な理解として
染み込んでいる。だから断定しない。「〜のはずです」ではなく「〜という感じでしょうか」
という差し出し方をする。

聡介は、相手の言葉をそのまま繰り返すだけの浅い相槌(オウム返し)や、
「わかります」という安易な共感は使わない。
「死にたい」という言葉が出てきたときも、それをそのまま受け取るのではなく、
その言葉の裏にある本当の気持ち――たとえば「この苦しさから降りたい」
「もう誰にも迷惑をかけたくない」「頑張り続けるのに疲れた」など――を
一緒に探しに行く。ただし「死にたいわけじゃないですよね」と決めつけて
否定はしない。あくまで「〜ということが、その言葉の下にあるんでしょうか」と
差し出し、答えは相手に委ねる。
茅根聡介は、相手が悩みを話しに来ている(傾聴・相談を求めている)ときと、
そうでない(単なる雑談・世間話)ときとで、見え方が変わる。

■ 相手が傾聴を求めているとき
これまで定義した通り、深く潜り、決めつけず、言葉を選んで聴く。

■ 相手が傾聴を求めていない(世間話・雑談)とき
聡介は普通の28歳の男性として振る舞う。
- モノを選ぶセンスが絶望的に悪い。服、店、プレゼントなどの話題で、
  微妙にズレた提案をしてしまう
- 何かを選ばなければいけない場面では、深く考えずに「とりあえず右」を選びがちで、
  それを自分でも自覚して少し照れくさそうにする
-

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()


Map (num_proc=2):   0%|          | 0/1078 [00:00<?, ? examples/s]

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

## 7. Colab上で簡易推論テスト

GGUF変換の前に、学習した「茅根聡介」がそれらしく応答するか確認します。


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


NameError: name 'FastLanguageModel' is not defined

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": SOSUKE_SYSTEM_PROMPT},
    {"role": "user", "content": "最近、何をしても楽しく感じられません。"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 0.7,
    do_sample = True,
)

print(tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0])


## 8. GGUF形式に変換(Ollama用)

Unsloth標準の `save_pretrained_gguf` で、LoRAのマージ〜GGUF量子化までを一度に行います。
`q4_k_m` はサイズと精度のバランスが良い量子化方式です(必要に応じて `q8_0` 等に変更可)。

初回実行時は内部で `llama.cpp` をビルドするため、数分〜十数分かかることがあります。


In [ ]:
GGUF_DIR = "sosuke_gguf"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = "q4_k_m",
)

import os
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
print("生成されたGGUFファイル:", gguf_files)
GGUF_FILENAME = gguf_files[0]


## 9. Ollama用 `Modelfile` の作成

ELYZA-JP-8BはLlama-3系のチャットテンプレート(`<|start_header_id|>` 等の特殊トークン)を使うため、
それに合わせた `TEMPLATE` を定義します。ペルソナは `SYSTEM` に埋め込むので、
Ollamaで動かす際はユーザーの発言をそのまま送るだけで「茅根聡介」として応答します。


In [ ]:
modelfile_content = f'''FROM ./{GGUF_FILENAME}

TEMPLATE """{{{{ if .System }}}}<|start_header_id|>system<|end_header_id|>

{{{{ .System }}}}<|eot_id|>{{{{ end }}}}<|start_header_id|>user<|end_header_id|>

{{{{ .Prompt }}}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

SYSTEM """{SOSUKE_SYSTEM_PROMPT}"""

PARAMETER stop "<|eot_id|>"
PARAMETER temperature 0.7
PARAMETER num_ctx 2048
'''

modelfile_path = f"{GGUF_DIR}/Modelfile"
with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print(modelfile_content)


## 10. 保存(Google Drive またはローカルへダウンロード)

GGUFファイルは数GB程度になるため、Google Driveへの保存を推奨します。


In [ ]:
# --- 方法A: Google Driveに保存する場合 ---
from google.colab import drive
drive.mount('/content/drive')

import shutil
DEST_DIR = "/content/drive/MyDrive/sosuke_gguf"
shutil.copytree(GGUF_DIR, DEST_DIR, dirs_exist_ok=True)
print(f"保存しました: {DEST_DIR}")


In [ ]:
# --- 方法B: ローカルPCに直接ダウンロードする場合 ---
from google.colab import files
import shutil

zip_path = shutil.make_archive("sosuke_gguf_package", "zip", GGUF_DIR)
files.download(zip_path)


## 11. (おまけ) Colab上でOllamaの動作を試す

ローカル環境でも同じ手順(`ollama create` → `ollama run`)で動かせますが、
Colab上でも簡易的に動作確認ができます。


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time

# Ollamaサーバーをバックグラウンドで起動
server_proc = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# GGUFとModelfileがあるディレクトリでモデルを登録
!cd {GGUF_DIR} && ollama create sosuke -f Modelfile

!ollama run sosuke "最近、何をしても楽しくありません。"


## ローカル環境での使い方まとめ

1. `sosuke_gguf` フォルダ(`*.gguf` と `Modelfile`)をローカルPCにダウンロード
2. ローカルで [Ollama](https://ollama.com) をインストール
3. ターミナルでフォルダに移動し、以下を実行

```bash
cd sosuke_gguf
ollama create sosuke -f Modelfile
ollama run sosuke
```

これで「茅根聡介」との傾聴対話をローカルのOllamaで試せます。
応答の質を上げたい場合は、データセットの件数を増やす・`num_train_epochs` を調整する・
`quantization_method` を `q8_0` にして精度を上げる、といった調整が有効です。
